# Cat vs Dog Image Classification — Complete Capstone Notebook

This notebook follows the supplied capstone brief: dataset audit, cleaning, preprocessing, traditional ML, HOG/PCA, CNN, augmentation, MobileNetV2 transfer learning, evaluation, and saving a model for the Streamlit website.

**Dataset requirement:** a ZIP file containing two class folders named `Cat` and `Dog` (capitalization does not matter). The notebook also searches recursively for folders named `cat`/`cats` and `dog`/`dogs`.

Run the cells from top to bottom in Google Colab. Do not skip the dataset upload and training cells.

## 1. Install and import libraries

Run this cell first. Colab already includes many libraries; this installs any missing packages.

In [ ]:
!pip -q install scikit-image seaborn

In [ ]:
import os, zipfile, shutil, random, hashlib, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from skimage.feature import hog

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Connect Google Drive and locate your dataset

First upload your dataset ZIP to Google Drive. You can also extract it in Drive yourself.
This cell mounts Drive and lets you choose either:
- a ZIP file stored in Drive (the notebook extracts it once into Colab's temporary storage), or
- an already-extracted folder in Drive containing the Cat and Dog class folders.

Edit `DRIVE_DATASET_PATH` to match the exact path and name in your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile

# OPTION A: ZIP file in Google Drive
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/Cats and Dogs.zip')

# OPTION B: If you already extracted the dataset in Drive, use its folder path instead.
# Example:
# DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/Cats and Dogs')

if not DRIVE_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset path not found: {DRIVE_DATASET_PATH}. "
        "Edit DRIVE_DATASET_PATH to the correct name/location in MyDrive."
    )

if DRIVE_DATASET_PATH.is_file() and DRIVE_DATASET_PATH.suffix.lower() == '.zip':
    EXTRACT_DIR = Path('/content/cat_dog_dataset')
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("Extracting dataset from Google Drive ZIP. This is a one-time step per Colab runtime...")
    with zipfile.ZipFile(DRIVE_DATASET_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    ROOT = EXTRACT_DIR
elif DRIVE_DATASET_PATH.is_dir():
    ROOT = DRIVE_DATASET_PATH
else:
    raise ValueError("The selected path must be a .zip file or an extracted dataset folder.")

print("Dataset location:", ROOT)
print("Drive connected successfully.")

**Note:** The main dataset cell above already connects Google Drive. You do not need a separate upload cell.

In [ ]:
# No additional Drive setup is needed. Drive is mounted in Section 2.\n

## 3. Locate class folders and create image metadata

The scan records the image path, class, width, height, channels and file extension. Corrupted/unreadable files are recorded separately.

In [ ]:
# ROOT was set in Section 2, based on your Google Drive ZIP or extracted folder.
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

cat_files, dog_files = [], []
for folder in ROOT.rglob("*"):
    if folder.is_dir():
        name = folder.name.lower()
        if name in {"cat", "cats"}:
            cat_files.extend([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
        elif name in {"dog", "dogs"}:
            dog_files.extend([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

cat_files = sorted(set(cat_files))
dog_files = sorted(set(dog_files))
print("Cat files found:", len(cat_files))
print("Dog files found:", len(dog_files))
if not cat_files or not dog_files:
    raise ValueError("Could not find both class folders. Ensure the dataset contains Cat and Dog folders.")

records, bad_files = [], []
for label, paths in [(0, cat_files), (1, dog_files)]:
    for p in paths:
        try:
            with Image.open(p) as im:
                im.verify()
            with Image.open(p) as im:
                w, h = im.size
                channels = len(im.getbands())
            records.append({"path":str(p), "label":label,
                            "class_name":"Cat" if label == 0 else "Dog",
                            "width":w, "height":h, "channels":channels,
                            "extension":p.suffix.lower(),
                            "file_size_kb":round(p.stat().st_size/1024, 2)})
        except Exception as e:
            bad_files.append({"path":str(p), "error":str(e)})

df = pd.DataFrame(records)
bad_df = pd.DataFrame(bad_files)
print("Valid images:", len(df), "| Problematic files:", len(bad_df))
display(df.head())
display(df["class_name"].value_counts())

In [ ]:
if len(bad_df):
    display(bad_df.head(10))
    bad_df.to_csv("/content/corrupted_images.csv", index=False)
    print("Corrupted file list saved to /content/corrupted_images.csv")

counts = df["class_name"].value_counts()
percentages = (counts / counts.sum() * 100).round(2)
display(pd.DataFrame({"count":counts, "percentage":percentages}))
print("Extensions:", sorted(df.extension.unique()))
display(df[["width","height","channels","file_size_kb"]].describe())

## 4. Explore and visualize images

The plots show class balance, image dimensions and random example images.

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df, x="class_name")
plt.title("Number of Cat and Dog Images")
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df["width"], bins=30, ax=ax[0]); ax[0].set_title("Image widths")
sns.histplot(df["height"], bins=30, ax=ax[1]); ax[1].set_title("Image heights")
plt.show()

sample_df = df.sample(min(20, len(df)), random_state=SEED)
fig, axes = plt.subplots(4, 5, figsize=(14,10))
for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    with Image.open(row.path) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(row.class_name); ax.axis("off")
for ax in axes.flat[len(sample_df):]: ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Remove exact duplicate files and create train/validation/test splits

We use image file hashes to identify exact duplicates. The split is stratified so both classes keep similar proportions in each set. The test set is held back until final evaluation.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

df["sha256"] = df["path"].apply(file_sha256)
before = len(df)
df = df.drop_duplicates("sha256").reset_index(drop=True)
print("Exact duplicate files removed:", before-len(df))
print("Images after cleaning:", len(df))

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"])

print("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))
print("Train class counts:\n", train_df.class_name.value_counts())
print("Validation class counts:\n", val_df.class_name.value_counts())
print("Test class counts:\n", test_df.class_name.value_counts())

split_dir = Path("/content/split_files")
split_dir.mkdir(exist_ok=True)
train_df.to_csv(split_dir/"train.csv", index=False)
val_df.to_csv(split_dir/"validation.csv", index=False)
test_df.to_csv(split_dir/"test.csv", index=False)

## 6. Image preprocessing

For the deep-learning models, images are converted to RGB and resized to 160×160. The `tf.data` pipeline loads images in batches, which avoids keeping the entire dataset in RAM.

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.float32)

def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.label.values))
    if training:
        ds = ds.shuffle(min(len(frame), 5000), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

for xb, yb in train_ds.take(1):
    print("Batch image shape:", xb.shape)
    print("Pixel range:", float(tf.reduce_min(xb)), "to", float(tf.reduce_max(xb)))

## 7. Traditional ML baselines using flattened pixels

Traditional ML on raw pixels is memory-intensive. To keep Colab practical, this section samples a limited number of images and uses 32×32 RGB pixels. Increase `ML_MAX_PER_CLASS` if your runtime has enough memory.

In [ ]:
ML_SIZE = 32
ML_MAX_PER_CLASS = 1500

def load_small_pixels(frame, size=ML_SIZE, max_per_class=ML_MAX_PER_CLASS):
    parts = []
    for label in [0,1]:
        sub = frame[frame.label == label]
        if len(sub) > max_per_class:
            sub = sub.sample(max_per_class, random_state=SEED)
        parts.append(sub)
    use_df = pd.concat(parts).sample(frac=1, random_state=SEED)
    X, y = [], []
    for row in use_df.itertuples():
        with Image.open(row.path) as im:
            im = im.convert("RGB").resize((size,size))
            X.append(np.asarray(im, dtype=np.float32).reshape(-1)/255.0)
        y.append(row.label)
    return np.asarray(X, dtype=np.float32), np.asarray(y), use_df

Xtr, ytr, ml_train_used = load_small_pixels(train_df)
Xte, yte, ml_test_used = load_small_pixels(test_df, max_per_class=500)
print("Traditional ML feature shape:", Xtr.shape)
print("64x64 RGB dimensionality would be:", 64*64*3)

In [ ]:
traditional_models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "SVM - Raw Pixels": make_pipeline(StandardScaler(), LinearSVC(random_state=SEED)),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1)
}
traditional_results = []
for name, model in traditional_models.items():
    start = time.time()
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    row = {"Model":name, "Accuracy":accuracy_score(yte,pred),
           "Precision":precision_score(yte,pred,zero_division=0),
           "Recall":recall_score(yte,pred,zero_division=0),
           "F1":f1_score(yte,pred,zero_division=0),
           "Training_seconds":round(time.time()-start,2)}
    traditional_results.append(row)
    print(name, row)
traditional_results_df = pd.DataFrame(traditional_results)
display(traditional_results_df)

## 8. HOG features and PCA

HOG describes image edges and shapes. PCA reduces the number of features while retaining as much variation as possible. This section uses a smaller balanced sample to keep computation manageable.

In [ ]:
HOG_MAX_PER_CLASS = 700
def load_hog_features(frame, max_per_class=HOG_MAX_PER_CLASS):
    selected = pd.concat([
        frame[frame.label==c].sample(min(max_per_class, (frame.label==c).sum()), random_state=SEED)
        for c in [0,1]
    ]).sample(frac=1, random_state=SEED)
    feats, labels = [], []
    for row in selected.itertuples():
        with Image.open(row.path) as im:
            im = im.convert("RGB").resize((64,64))
            arr = np.asarray(im)
        feat = hog(arr, orientations=9, pixels_per_cell=(8,8),
                   cells_per_block=(2,2), channel_axis=-1)
        feats.append(feat); labels.append(row.label)
    return np.asarray(feats), np.asarray(labels)

Xhog, yhog = load_hog_features(train_df)
Xhog_test, yhog_test = load_hog_features(test_df, max_per_class=250)
hog_svm = make_pipeline(StandardScaler(), LinearSVC(random_state=SEED))
hog_svm.fit(Xhog, yhog)
hog_pred = hog_svm.predict(Xhog_test)
print("HOG + SVM accuracy:", accuracy_score(yhog_test, hog_pred))

In [ ]:
# PCA demonstration on a manageable subset of flattened training images
pca_scaler = StandardScaler()
Xtr_scaled = pca_scaler.fit_transform(Xtr)
pca = PCA(n_components=0.90, svd_solver="full", random_state=SEED)
Xtr_pca = pca.fit_transform(Xtr_scaled)
print("Original features:", Xtr.shape[1])
print("Features retaining 90% variance:", Xtr_pca.shape[1])
print("Explained variance:", pca.explained_variance_ratio_.sum())

Xte_scaled = pca_scaler.transform(Xte)
Xte_pca = pca.transform(Xte_scaled)
pca_model = LogisticRegression(max_iter=500)
pca_model.fit(Xtr_pca, ytr)
pca_pred = pca_model.predict(Xte_pca)
print("PCA + Logistic Regression accuracy:", accuracy_score(yte, pca_pred))

## 9. Custom CNN

This CNN learns visual patterns such as edges, textures and shapes. We use early stopping and save the best validation model.

In [ ]:
cnn = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid")
])
cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
cnn.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("/content/custom_cnn_best.keras", monitor="val_loss", save_best_only=True)
]
history_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=callbacks)

In [ ]:
def plot_history(history, title):
    fig, ax = plt.subplots(1,2,figsize=(12,4))
    ax[0].plot(history.history["accuracy"], label="Train")
    ax[0].plot(history.history["val_accuracy"], label="Validation")
    ax[0].set_title(title+" Accuracy"); ax[0].legend()
    ax[1].plot(history.history["loss"], label="Train")
    ax[1].plot(history.history["val_loss"], label="Validation")
    ax[1].set_title(title+" Loss"); ax[1].legend()
    plt.show()
plot_history(history_cnn, "Custom CNN")

## 10. Data augmentation

Augmentation creates varied training examples using flips, rotations and zoom. It is applied only to training images.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.08, 0.08)
], name="data_augmentation")

aug_cnn = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    augmentation,
    layers.Conv2D(32, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"), layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid")
])
aug_cnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history_aug = aug_cnn.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=[
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("/content/augmented_cnn_best.keras", monitor="val_loss", save_best_only=True)
])
plot_history(history_aug, "CNN with Augmentation")

## 11. Transfer learning with MobileNetV2

MobileNetV2 is pretrained on ImageNet. We first freeze its feature-extraction layers and train a small Cat/Dog classification head.

In [ ]:
def mobilenet_preprocess(image, label):
    # Dataset images are currently [0,1]; MobileNetV2 expects [-1,1].
    image = image * 2.0 - 1.0
    return image, label

train_mn = train_ds.map(mobilenet_preprocess, num_parallel_calls=AUTOTUNE)
val_mn = val_ds.map(mobilenet_preprocess, num_parallel_calls=AUTOTUNE)
test_mn = test_ds.map(mobilenet_preprocess, num_parallel_calls=AUTOTUNE)

base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
mobilenet_model = keras.Model(inputs, outputs)
mobilenet_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                        loss="binary_crossentropy", metrics=["accuracy"])

history_mn = mobilenet_model.fit(train_mn, validation_data=val_mn, epochs=8, callbacks=[
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("/content/mobilenetv2_best.keras", monitor="val_loss", save_best_only=True)
])
plot_history(history_mn, "MobileNetV2")

### Optional fine-tuning

After the frozen-backbone baseline, unfreeze only the top portion of MobileNetV2 and train with a very small learning rate. This cell is optional if Colab runs out of time or memory.

In [ ]:
# Optional fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

mobilenet_model.compile(optimizer=keras.optimizers.Adam(1e-5),
                        loss="binary_crossentropy", metrics=["accuracy"])
history_fine = mobilenet_model.fit(train_mn, validation_data=val_mn, epochs=5, callbacks=[
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
])

## 12. Final evaluation

The final evaluation uses the untouched test dataset. Class encoding: Cat = 0 and Dog = 1.

In [ ]:
def evaluate_binary_model(model, dataset, name):
    y_true = np.concatenate([y.numpy().reshape(-1) for _,y in dataset]).astype(int)
    probs = model.predict(dataset, verbose=0).reshape(-1)
    y_pred = (probs >= 0.5).astype(int)
    try: auc = roc_auc_score(y_true, probs)
    except Exception: auc = float("nan")
    metrics = {
        "Model":name,
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,zero_division=0),
        "Recall":recall_score(y_true,y_pred,zero_division=0),
        "F1":f1_score(y_true,y_pred,zero_division=0),
        "ROC_AUC":auc
    }
    print(metrics)
    print(classification_report(y_true,y_pred,target_names=["Cat","Dog"]))
    cm = confusion_matrix(y_true,y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Cat","Dog"], yticklabels=["Cat","Dog"])
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(name+" Confusion Matrix")
    plt.show()
    return metrics, y_true, y_pred, probs

cnn_metrics, y_true_cnn, y_pred_cnn, prob_cnn = evaluate_binary_model(cnn, test_ds, "Custom CNN")
aug_metrics, y_true_aug, y_pred_aug, prob_aug = evaluate_binary_model(aug_cnn, test_ds, "Augmented CNN")
mn_metrics, y_true_mn, y_pred_mn, prob_mn = evaluate_binary_model(mobilenet_model, test_mn, "MobileNetV2")

In [ ]:
comparison = pd.DataFrame([cnn_metrics, aug_metrics, mn_metrics])
display(comparison.sort_values("F1", ascending=False))
comparison.to_csv("/content/model_comparison.csv", index=False)

## 13. Inspect incorrect predictions

This shows examples the final MobileNetV2 model misclassified. Review these cases and describe possible reasons in your report (lighting, pose, background, partial visibility, etc.).

In [ ]:
wrong = np.where(y_true_mn != y_pred_mn)[0]
print("Misclassified test images:", len(wrong))
test_paths = test_df.path.to_list()
# test_ds preserves test_df order because it is not shuffled.
for idx in wrong[:12]:
    with Image.open(test_paths[idx]) as im:
        plt.figure(figsize=(3,3)); plt.imshow(im.convert("RGB")); plt.axis("off")
    actual = "Dog" if y_true_mn[idx] == 1 else "Cat"
    predicted = "Dog" if y_pred_mn[idx] == 1 else "Cat"
    plt.title(f"Actual: {actual} | Predicted: {predicted} | Confidence: {prob_mn[idx]:.2%}")
    plt.show()

## 14. Save the model and supporting files

The website expects a saved model named `cat_dog_model.keras`. This notebook saves the trained MobileNetV2 model and a small metadata file. It also saves a copy of the class mapping.

In [ ]:
OUTPUT_DIR = Path("/content/cat_dog_output")
OUTPUT_DIR.mkdir(exist_ok=True)
mobilenet_model.save(OUTPUT_DIR/"cat_dog_model.keras")
with open(OUTPUT_DIR/"model_info.json","w") as f:
    json.dump({
        "model_name":"MobileNetV2",
        "image_size":IMG_SIZE,
        "class_mapping":{"0":"Cat","1":"Dog"},
        "preprocessing":"RGB resize to 160x160; pixel values 0-1 then transform to -1..1",
        "threshold":0.5
    }, f, indent=2)
comparison.to_csv(OUTPUT_DIR/"model_comparison.csv", index=False)

zip_out = shutil.make_archive("/content/cat_dog_model_files", "zip", OUTPUT_DIR)
print("Saved:", zip_out)
from google.colab import files
files.download(zip_out)

## 15. What to do after training

1. Download `cat_dog_model_files.zip` from the last cell.
2. Extract it on your computer.
3. Put `cat_dog_model.keras` in the `models` folder beside `app.py`.
4. Install the website requirements and run Streamlit using the instructions in `README.txt`.

**Important:** Training may take time. Use a Colab GPU (`Runtime → Change runtime type → T4 GPU`, if available). Run cells in order. Keep the test set untouched until final evaluation.